In [ ]:
!pip install transformers torch pandas numpy tqdm scikit-learn lightgbm --quiet
print("✅ Done")
!pip install tabulate

In [ ]:
CONFIG = {
    "MODEL_NAME":         "vinai/phobert-base-v2",
    "MAX_LENGTH":         128,
    "BATCH_SIZE":         16,
    "EPOCHS":             3,
    "LEARNING_RATE":      2e-5,
    "VECTOR_DIM":         768,
    "OUTPUT_DIR":         "./phobert_foodtour",

    "TRENDING_WEIGHT":    0.7,
    "TRENDING_THRESHOLD": 1.5,
    "ES_WEIGHT_MAX":      10000,
    "ES_WEIGHT_MIN":      1,
    "EPSILON":            1e-6,
}
print("✅ Config loaded")

In [ ]:
from google.colab import files

print("📁 Upload products.csv")
up = files.upload()
for fn, data in up.items():
    with open(fn, 'wb') as f: f.write(data)
    print(f"  ✅ {fn}")

print("\n📁 Upload search_analytics.csv")
up2 = files.upload()
for fn, data in up2.items():
    with open(fn, 'wb') as f: f.write(data)
    print(f"  ✅ {fn}")

📁 Upload products.csv


In [ ]:
import pandas as pd
import numpy as np
import json
import re
import random
from collections import Counter, defaultdict
from datetime import datetime, timedelta

# ===== REMOVE ACCENT =====
_VN_MAP = {
    'à':'a','á':'a','ả':'a','ã':'a','ạ':'a',
    'â':'a','ầ':'a','ấ':'a','ẩ':'a','ẫ':'a','ậ':'a',
    'ă':'a','ằ':'a','ắ':'a','ẳ':'a','ẵ':'a','ặ':'a',
    'è':'e','é':'e','ẻ':'e','ẽ':'e','ẹ':'e',
    'ê':'e','ề':'e','ế':'e','ể':'e','ễ':'e','ệ':'e',
    'ì':'i','í':'i','ỉ':'i','ĩ':'i','ị':'i',
    'ò':'o','ó':'o','ỏ':'o','õ':'o','ọ':'o',
    'ô':'o','ồ':'o','ố':'o','ổ':'o','ỗ':'o','ộ':'o',
    'ơ':'o','ờ':'o','ớ':'o','ở':'o','ỡ':'o','ợ':'o',
    'ù':'u','ú':'u','ủ':'u','ũ':'u','ụ':'u',
    'ư':'u','ừ':'u','ứ':'u','ử':'u','ữ':'u','ự':'u',
    'ỳ':'y','ý':'y','ỷ':'y','ỹ':'y','ỵ':'y','đ':'d',
}

def remove_vn_accent(text):
    if not text:
        return ''
    return ''.join(_VN_MAP.get(c, c) for c in text.lower())

# ===== 🔥 SPLIT CHUẨN THEO STRUCTURE =====
def split_messy_json(text):
    if not text:
        return [], [], []

    s = str(text)

    # tìm tất cả block JSON dạng [...]
    blocks = re.findall(r'\[(.*?)\]', s)

    def parse_block(block):
        items = re.findall(r'"(.*?)"', block)
        return items

    tags = parse_block(blocks[0]) if len(blocks) > 0 else []
    ingredients = parse_block(blocks[1]) if len(blocks) > 1 else []
    images = parse_block(blocks[2]) if len(blocks) > 2 else []

    return tags, ingredients, images

# ===== BUILD TEXT =====
def build_product_text(row):
    name = str(row.get('name') or '')
    desc = str(row.get('description') or '')

    tags = row.get('tags_str', '')
    ingredients = row.get('ingredients_str', '')
    category = str(row.get('category_name') or '')

    return f"{name} {desc} {tags} {ingredients} {category}".strip()

print("✅ Helpers ready")

In [ ]:
import csv
from tabulate import tabulate

rows = []

with open("products.csv", encoding="utf-8-sig") as f:
    reader = csv.reader(f)

    headers = next(reader)
    n_cols = len(headers)

    for i, row in enumerate(reader):
        try:
            # ===== FIX ROW VỠ =====
            if len(row) > n_cols:
                head = row[:5]
                tail = row[-(n_cols - 8):]

                middle = row[5: len(row) - len(tail)]
                merged = ' '.join(middle)

                row = head + [merged] + [''] * 2 + tail

            if len(row) < n_cols:
                row = row + [''] * (n_cols - len(row))

            if len(row) != n_cols:
                print(f"❌ Row lỗi {i}")
                continue

            rows.append(row)

        except Exception as e:
            print(f"❌ Error row {i}: {row}")
            raise e

# ===== DATAFRAME =====
df_products = pd.DataFrame(rows, columns=headers)
df_products.columns = [c.strip().lower().replace(' ', '_') for c in df_products.columns]

# ===== CLEAN =====
df_products['description']   = df_products.get('description', '').fillna('')
df_products['tags']          = df_products.get('tags', '').fillna('')
df_products['category_name'] = df_products.get('category_name', '').fillna('')

# ===== 🔥 SPLIT ĐÚNG =====
df_products[['tags_clean','ingredients_clean','images_clean']] = df_products['tags'].apply(
    lambda x: pd.Series(split_messy_json(x))
)

# ===== STRING =====
df_products['tags_str'] = df_products['tags_clean'].apply(lambda x: ' '.join(x))
df_products['ingredients_str'] = df_products['ingredients_clean'].apply(lambda x: ' '.join(x))
df_products['images_str'] = df_products['images_clean'].apply(lambda x: ' '.join(x))

# ===== BUILD TEXT =====
df_products['product_text'] = df_products.apply(build_product_text, axis=1)

# ===== TABLE FULL =====
cols = [
    'id','name','price','discount_price','rating','total_reviews',
    'shop_name','shop_city','category_name',
    'tags_str','ingredients_str','images_str'
]

print("\n✅ DONE:", len(df_products))

print("\n=== FULL TABLE ===")
print(tabulate(
    df_products[cols].head(8),
    headers='keys',
    tablefmt='fancy_grid',
    showindex=False
))

print("\n=== SAMPLE TEXT ===")
print(df_products['product_text'].iloc[0])

In [ ]:
df_analytics = pd.read_csv("search_analytics.csv", encoding='utf-8-sig')
df_analytics.columns = [c.strip().lower().replace(' ', '_') for c in df_analytics.columns]
df_analytics['query_text']  = df_analytics.get('query_text', pd.Series(dtype=str)).fillna('')
df_analytics['searched_at'] = pd.to_datetime(df_analytics.get('searched_at'), errors='coerce')
df_analytics = df_analytics[df_analytics['query_text'].str.strip() != '']

print(f"✅ {len(df_analytics)} analytics rows")
print(f"Clicks: {df_analytics['clicked_product_id'].notna().sum()}")
print(f"Columns: {list(df_analytics.columns)}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_NAME"])
model     = AutoModel.from_pretrained(CONFIG["MODEL_NAME"]).to(device)
print(f"✅ PhoBERT loaded: {CONFIG['MODEL_NAME']}")

def mean_pooling(model_output, attention_mask):
    token_emb    = model_output.last_hidden_state
    mask_expanded = attention_mask.unsqueeze(-1).expand(token_emb.size()).float()
    return torch.sum(token_emb * mask_expanded, 1) / torch.clamp(mask_expanded.sum(1), min=1e-9)

def get_embedding(text, model, tokenizer, device):
    enc = tokenizer(
        text, max_length=CONFIG['MAX_LENGTH'],
        padding='max_length', truncation=True, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        out = model(**enc)
    emb = mean_pooling(out, enc['attention_mask'])
    emb = nn.functional.normalize(emb, p=2, dim=1)
    return emb.cpu().numpy()[0].tolist()

In [ ]:
import random
from torch.utils.data import Dataset, DataLoader

# Positive pairs: query → clicked product
df_clicks = df_analytics[df_analytics['clicked_product_id'].notna()].copy()
df_clicks['clicked_product_id'] = df_clicks['clicked_product_id'].astype(int)

product_text_map = dict(zip(
    df_products['id'].astype(int),
    df_products['product_text']
))
product_id_list = list(product_text_map.keys())

pairs = []
for _, row in df_clicks.iterrows():
    query = str(row['query_text']).strip()
    pid   = int(row['clicked_product_id'])
    if pid in product_text_map and query:
        pairs.append({
            'query':        query,
            'product_text': product_text_map[pid],
            'label':        1.0
        })

# Negative pairs: random product không click
neg_pairs = []
for p in pairs:
    neg_id = random.choice(product_id_list)
    neg_pairs.append({
        'query':        p['query'],
        'product_text': product_text_map[neg_id],
        'label':        0.0
    })

all_pairs = pairs + neg_pairs
random.shuffle(all_pairs)
print(f"✅ {len(all_pairs)} pairs ({len(pairs)} positive, {len(neg_pairs)} negative)")


class SearchDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_len):
        self.pairs     = pairs
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        p     = self.pairs[idx]
        q_enc = self.tokenizer(
            p['query'], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        d_enc = self.tokenizer(
            p['product_text'], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'q_input_ids':      q_enc['input_ids'].squeeze(),
            'q_attention_mask': q_enc['attention_mask'].squeeze(),
            'd_input_ids':      d_enc['input_ids'].squeeze(),
            'd_attention_mask': d_enc['attention_mask'].squeeze(),
            'label':            torch.tensor(p['label'], dtype=torch.float),
        }

dataset    = SearchDataset(all_pairs, tokenizer, CONFIG['MAX_LENGTH'])
dataloader = DataLoader(dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True)
print(f"✅ DataLoader ready: {len(dataloader)} batches")

In [ ]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=CONFIG['LEARNING_RATE'])
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=len(dataloader) * CONFIG['EPOCHS']
)
loss_fn = nn.CosineEmbeddingLoss()

model.train()
for epoch in range(CONFIG['EPOCHS']):
    total_loss = 0
    for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{CONFIG['EPOCHS']}"):
        q_ids  = batch['q_input_ids'].to(device)
        q_mask = batch['q_attention_mask'].to(device)
        d_ids  = batch['d_input_ids'].to(device)
        d_mask = batch['d_attention_mask'].to(device)
        labels = batch['label'].to(device)

        q_out = model(input_ids=q_ids, attention_mask=q_mask)
        d_out = model(input_ids=d_ids, attention_mask=d_mask)
        q_emb = mean_pooling(q_out, q_mask)
        d_emb = mean_pooling(d_out, d_mask)
        q_emb = nn.functional.normalize(q_emb, p=2, dim=1)
        d_emb = nn.functional.normalize(d_emb, p=2, dim=1)

        target = torch.where(labels > 0.5,
                             torch.ones_like(labels),
                             torch.full_like(labels, -1.0))
        loss = loss_fn(q_emb, d_emb, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} — Avg Loss: {total_loss/len(dataloader):.4f}")

print("✅ Training complete!")

In [ ]:
import os, shutil
from google.colab import files

os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)
model.save_pretrained(CONFIG['OUTPUT_DIR'])
tokenizer.save_pretrained(CONFIG['OUTPUT_DIR'])
print(f"✅ Model saved to {CONFIG['OUTPUT_DIR']}")

shutil.make_archive("phobert_foodtour", 'zip', CONFIG['OUTPUT_DIR'])
files.download("phobert_foodtour.zip")
print("✅ Downloaded phobert_foodtour.zip")

In [ ]:
model.eval()
print("Generating embeddings for all products...")

all_embeddings = []
for _, row in tqdm(df_products.iterrows(), total=len(df_products)):
    emb = get_embedding(row['product_text'], model, tokenizer, device)
    all_embeddings.append(emb)

df_products['embedding'] = all_embeddings
print(f"✅ {len(all_embeddings)} embeddings generated, dim={len(all_embeddings[0])}")

In [ ]:
# ==================== CELL 12 - TRENDING ĐÃ SỬA (PHÙ HỢP VỚI DỮ LIỆU CỦA BẠN) ====================
from datetime import datetime, timedelta
from collections import defaultdict, Counter

def normalize_query(text):
    if not text:
        return ''
    text = text.lower().strip()
    text = remove_vn_accent(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

now = df_analytics['searched_at'].max().replace(tzinfo=None)
print(f"📅 Now (from data): {now.date()}")

# === ĐIỀU CHỈNH CHO DỮ LIỆU THỰC TẾ (chỉ ~10 ngày) ===
cut_1d  = now - timedelta(days=1)
cut_7d  = now - timedelta(days=7)
cut_15d = now - timedelta(days=15)   # ← thay vì 30 ngày

print(f"Cut 1d : {cut_1d.date()}")
print(f"Cut 7d : {cut_7d.date()}")
print(f"Cut 15d: {cut_15d.date()}\n")

stats = defaultdict(lambda: {
    'texts': [], 'total': 0,
    'c1d': 0, 'c7d': 0, 'c15d': 0, 'last': None
})

for _, row in df_analytics.iterrows():
    raw_q = str(row.get('query_text', '')).strip()
    if not raw_q:
        continue

    norm = normalize_query(raw_q)
    ts = row.get('searched_at')
    if pd.isna(ts):
        ts = now

    s = stats[norm]
    s['texts'].append(raw_q)
    s['total'] += 1

    if ts >= cut_1d:  s['c1d'] += 1
    if ts >= cut_7d:  s['c7d'] += 1
    if ts >= cut_15d: s['c15d'] += 1   # dùng 15 ngày thay vì 30

    if s['last'] is None or ts > s['last']:
        s['last'] = ts

# ====================== TÍNH TRENDING ======================
suggestion_docs = []
epsilon = CONFIG.get('EPSILON', 1e-6)

for norm, s in stats.items():
    recent_count = s['c15d']                      # dùng 15 ngày
    avg_recent = recent_count / 15.0 if recent_count > 0 else 0

    # Tính multiplier dựa trên search gần đây so với trung bình
    if avg_recent < epsilon:
        multiplier = 1.0
    else:
        multiplier = (s['c1d'] + s['c7d']*0.5) / (avg_recent + epsilon)   # ưu tiên search rất mới

    final_score = recent_count * (1 + CONFIG['TRENDING_WEIGHT'] * (multiplier - 1))
    final_score = max(final_score, 0.0)

    best_text = Counter(s['texts']).most_common(1)[0][0]

    suggestion_docs.append({
        'normalized': norm,
        'queryText': best_text,
        'queryNormalized': norm,
        'noAccent': remove_vn_accent(best_text),
        'totalSearches': s['total'],
        'searches1d': s['c1d'],
        'searches7d': s['c7d'],
        'searches15d': s['c15d'],
        'avgDaily15d': round(avg_recent, 4),
        'trendingMultiplier': round(multiplier, 4),
        'finalScore': round(final_score, 4),
        'isTrending': multiplier >= CONFIG['TRENDING_THRESHOLD'],
        'lastSearchedAt': s['last'].strftime('%Y-%m-%d') if s['last'] else '',
    })

# Tính esWeight
if suggestion_docs:
    max_score = max(d['finalScore'] for d in suggestion_docs)
    for d in suggestion_docs:
        nw = d['finalScore'] / max_score if max_score > 0 else 0
        d['esWeight'] = max(CONFIG['ES_WEIGHT_MIN'],
                            min(CONFIG['ES_WEIGHT_MAX'],
                                int(CONFIG['ES_WEIGHT_MIN'] + nw * (CONFIG['ES_WEIGHT_MAX'] - CONFIG['ES_WEIGHT_MIN']))))

print(f"\n✅ {len(suggestion_docs)} suggestions built\n")

# In top 15
top15 = sorted(suggestion_docs, key=lambda x: x['esWeight'], reverse=True)[:15]
for d in top15:
    badge = "🔥" if d.get('isTrending', False) else "  "
    print(f"{badge} [{d['esWeight']:5d}] {d['queryText']:28} | total={d['totalSearches']:3d} | 15d={d['searches15d']:3d} | mult={d['trendingMultiplier']:.2f}")

In [ ]:
import lightgbm as lgb
import pickle
from sklearn.model_selection import train_test_split

# Fix: key phải là int để match với int(pid)
product_map = df_products.set_index(df_products['id'].astype(int)).to_dict('index')
product_id_list = list(product_map.keys())  # list of int

features, labels = [], []
for _, row in df_analytics.iterrows():
    pid = row.get('clicked_product_id')
    if pd.isna(pid): continue
    pid = int(float(pid))
    if pid not in product_map: continue
    p = product_map[pid]

    # Positive
    features.append([
        float(p.get('rating', 0) or 0),
        int(p.get('total_reviews', 0) or 0),
        float(p.get('price', 0) or 0),
        int(bool(p.get('is_available', 1))),
        int(row.get('result_count', 0) or 0),
        1.0 / max(int(row.get('result_count', 1) or 1), 1),
        int(row.get('click_position', 99) or 99),
    ])
    labels.append(1)

    # Negative: random product khác pid
    neg_id = pid
    while neg_id == pid:
        neg_id = random.choice(product_id_list)
    np_ = product_map[neg_id]
    features.append([
        float(np_.get('rating', 0) or 0),
        int(np_.get('total_reviews', 0) or 0),
        float(np_.get('price', 0) or 0),
        int(bool(np_.get('is_available', 1))),
        int(row.get('result_count', 0) or 0),
        0.0,
        99,
    ])
    labels.append(0)

print(f"✅ {len(features)} samples ({labels.count(1)} pos, {labels.count(0)} neg)")

X = np.array(features)
y = np.array(labels)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

ranking_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    class_weight='balanced',
    verbose=-1,
)
ranking_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(20)]
)

print(f"✅ Best iteration: {ranking_model.best_iteration_}")

with open("ranking_model.pkl", "wb") as f:
    pickle.dump(ranking_model, f)
files.download("ranking_model.pkl")
print("✅ Ranking model saved & downloaded")

In [ ]:
# Export embeddings ra CSV
import csv, json

with open("embeddings.csv", "w", newline='', encoding='utf-8-sig') as f:
    writer = csv.DictWriter(f, fieldnames=['product_id', 'embedding'])
    writer.writeheader()
    for _, row in df_products.iterrows():
        writer.writerow({
            'product_id': int(row['id']),
            'embedding':  json.dumps(row['embedding'])
        })

files.download("embeddings.csv")
print(f"✅ Exported {len(df_products)} embeddings")

In [ ]:
# ========== CELL 15: FLASK EMBEDDING SERVER ==========
!pip install flask -q

from flask import Flask, request, jsonify
import threading

embed_app = Flask(__name__)
model.eval()

@embed_app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

@embed_app.route("/embed", methods=["POST"])
def embed_endpoint():
    data = request.get_json()
    text = data.get("text", "").strip()
    if not text:
        return jsonify({"embedding": [0.0] * 768})
    enc = tokenizer(
        text, max_length=128,
        padding='max_length', truncation=True, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        out = model(**enc)
    emb = mean_pooling(out, enc['attention_mask'])
    emb = nn.functional.normalize(emb, p=2, dim=1)
    return jsonify({"embedding": emb.cpu().numpy()[0].tolist()})

t = threading.Thread(target=lambda: embed_app.run(
    host='0.0.0.0', port=5001, threaded=False, use_reloader=False
))
t.daemon = True
t.start()

import time
time.sleep(2)
print("✅ Flask server running on port 5001")

In [ ]:
# ========== CELL 16: CLOUDFLARE TUNNEL ==========
import subprocess, time, re

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:5001'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

tunnel_url = None
print("⏳ Đang tạo tunnel...")

for _ in range(60):
    line = proc.stdout.readline().decode('utf-8', errors='ignore')
    if line.strip():
        print(line.strip())
    match = re.search(
        r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break
    time.sleep(1)

if tunnel_url:
    print(f"\n{'='*60}")
    print(f"🌐 TUNNEL URL: {tunnel_url}")
    print(f"{'='*60}")
    print(f"\n👉 Dán vào application.properties:")
    print(f"   phobert.server-url={tunnel_url}")
else:
    print("❌ Không lấy được URL, chạy lại cell này")

In [ ]:
# ========== CELL 17: TEST ==========
import requests

resp = requests.get(f"{tunnel_url}/health", timeout=15)
print(f"Health: {resp.json()}")

resp2 = requests.post(
    f"{tunnel_url}/embed",
    json={"text": "phở bò"},
    timeout=30
)
emb = resp2.json()['embedding']
print(f"✅ Embedding dim={len(emb)}, sample={emb[:3]}")